### 정연 EDA - 길단위 인구

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 깨짐 방지

In [ ]:
from pathlib import Path

current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent

csv_path = project_root / 'data' / '05_길단위인구' / '서울시 상권분석서비스(길단위인구-상권).csv'

df = pd.read_csv(csv_path, encoding='cp949')
print(f'✅ 로드 완료: {df.shape}')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 깨짐 방지

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.head(5)

In [ ]:
# 결측치 확인
print(df.isnull().sum())

### 결측치, 중복 x

In [ ]:
# 기초 통계량 확인 (평균, 최소, 최대값 등)
df.describe()

# 총 유동인구: 데이터 스케일 차이가 너무 크다 → 로그 변환
# 연령대: 모든 연령대에 0이 존재 
# 요일: max값 - 토,일 100% → 특정 요일에 유동인구가 거의 몰린 상권이 존재

In [ ]:
import numpy as np
df['log_유동인구'] = np.log1p(df['총_유동인구_수'])
df['log_유동인구']

In [ ]:
sns.histplot(df['log_유동인구'])

## 유동 인구 로그 변환 후 확인
## 원래 총_유동인구_수는: 작은 값 많고 큰 값 일부 (관광특구 같은 애들)
## 로그 변환 후 확인 -> 종 모양에 가까움을 확인 

## 상권 유형별 성별 유동인구 비율
### 여성이 남성보다 구매가 많음을 확인 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 성별 비율
df['남성_비율'] = df['남성_유동인구_수'] / df['총_유동인구_수'] * 100
df['여성_비율'] = df['여성_유동인구_수'] / df['총_유동인구_수'] * 100

# 연령대 비율 (6개 컬럼)
age_cols = ['연령대_10','연령대_20','연령대_30','연령대_40','연령대_50','연령대_60_이상']
raw_cols = [f'{c}_유동인구_수' for c in age_cols]

for c, r in zip(age_cols, raw_cols):
    df[f'{c}_비율'] = df[r] / df['총_유동인구_수'] * 100

In [ ]:
# 상권 유형별 성별 비율 평균
gender_by_type = df.groupby('상권_구분_코드_명')[['남성_비율','여성_비율']].mean()

In [ ]:
df['남성_비율'] = df['남성_유동인구_수'] / df['총_유동인구_수'] * 100
df['여성_비율'] = df['여성_유동인구_수'] / df['총_유동인구_수'] * 100

age_cols = ['연령대_10','연령대_20','연령대_30','연령대_40','연령대_50','연령대_60_이상']

for col in age_cols:
    df[f'{col}_비율'] = df[f'{col}_유동인구_수'] / df['총_유동인구_수'] * 100

gender_by_type = df.groupby('상권_구분_코드_명')[['남성_비율','여성_비율']].mean()

import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

gender_by_type.plot(kind='bar', figsize=(8,5))

plt.title('상권 유형별 성별 유동인구 비율')
plt.ylabel('비율(%)')
plt.xticks(rotation=0)
plt.legend(['남성','여성'],
           loc='upper center',
           bbox_to_anchor=(0.5, -0.15),
           ncol=2)
plt.show()

### 상권 유형별 분기별 평균 유동인구 추세

In [ ]:
# 20211 → 연도 2021, 분기 1
df['연도'] = df['기준_년분기_코드'] // 10
df['분기'] = df['기준_년분기_코드'] % 10

# 시각화·정렬용 문자열 레이블
df['년분기'] = df['연도'].astype(str) + '-Q' + df['분기'].astype(str)

# pandas Period로 변환 (시계열 연산에 유리)
df['period'] = pd.PeriodIndex(
    df['연도'].astype(str) + 'Q' + df['분기'].astype(str), freq='Q'
)

# 정렬 확인
print(sorted(df['년분기'].unique()))
# ['2021-Q1', '2021-Q2', ..., '2023-Q4']

In [ ]:
df['연도'] = df['기준_년분기_코드'] // 10
df['분기'] = df['기준_년분기_코드'] % 10

df['년분기'] = df['연도'].astype(str) + '-Q' + df['분기'].astype(str)

ts = df.groupby(['년분기', '상권_구분_코드_명'])['총_유동인구_수'].mean().reset_index()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12,6))
fig, ax1 = plt.subplots(figsize=(12,6))
ax2 = ax1.twinx()  # 오른쪽 축

# 색상 (가독성 좋게)
colors = {
    '골목상권': '#7b6fd0',
    '발달상권': '#2ca02c',
    '전통시장': '#f39c12',
    '관광특구': '#1f77b4'
}

for name, grp in ts.groupby('상권_구분_코드_명'):
    
    if name == '관광특구':
        ax2.plot(grp['년분기'], grp['총_유동인구_수'],
                 marker='o', linewidth=2.5,
                 color=colors[name], label=name)
    else:
        ax1.plot(grp['년분기'], grp['총_유동인구_수'],
                 marker='o', linewidth=2,
                 color=colors[name], label=name)

# 축 설정
ax1.set_xlabel('분기')
ax1.set_ylabel('평균 유동인구 수 (골목·발달·전통)')
ax2.set_ylabel('평균 유동인구 수 (관광특구)', color=colors['관광특구'])

# x축 회전
ax1.tick_params(axis='x', rotation=45)

# y축 색상 맞추기
ax2.tick_params(axis='y', colors=colors['관광특구'])

# 숫자 포맷 (천 단위 콤마)
import matplotlib.ticker as mticker
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# 범례 합치기
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='upper center',
           bbox_to_anchor=(0.5, -0.2),
           ncol=4)

plt.title('상권 유형별 분기별 평균 유동인구 추세')

plt.tight_layout()
plt.show()

# 상권 유형별 시간대 유동인구 비율

# 골목상권 — 심야형
# 25.5%로 전체 상권 중 1위 → 심야 유흥·음식점 업종이 구조적으로 많다

# 관광특구
# 외국인 관광객은 특정 시간에 몰리지 않고 낮 내내 분산 방문

# 발달상권 - 직장인 루트
# 출근길 방문과 퇴근 후

# 전통시장 - 새벽·오전형
# 00~06시(21.7%)가 골목상권 다음으로 높음 
# → 새벽 도매·납품 상인의 이동 인구가 반영

In [ ]:
time_cols_raw = [
    '시간대_00_06_유동인구_수', '시간대_06_11_유동인구_수',
    '시간대_11_14_유동인구_수', '시간대_14_17_유동인구_수',
    '시간대_17_21_유동인구_수', '시간대_21_24_유동인구_수'
]
time_labels = ['00~06시', '06~11시', '11~14시', '14~17시', '17~21시', '21~24시']

# 비율 컬럼 생성
for col, label in zip(time_cols_raw, time_labels):
    df[f'{label}_비율'] = df[col] / df['총_유동인구_수'] * 100

time_ratio_cols = [f'{l}_비율' for l in time_labels]

In [ ]:
time_by_type = df.groupby('상권_구분_코드_명')[time_ratio_cols].mean()
time_by_type.columns = time_labels

colors = {
    '골목상권': '#7F77DD', '관광특구': '#378ADD',
    '발달상권': '#1D9E75', '전통시장': '#EF9F27'
}

fig, ax = plt.subplots(figsize=(10, 5))

for type_name, row in time_by_type.iterrows():
    ax.plot(time_labels, row.values,
            marker='o', markersize=6, linewidth=2,
            color=colors[type_name], label=type_name)

    # 피크 시간대 강조
    peak_idx = row.values.argmax()
    ax.annotate(f'{row.values[peak_idx]:.1f}%',
                xy=(peak_idx, row.values[peak_idx]),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=9, color=colors[type_name])

ax.set_title('상권 유형별 시간대 유동인구 비율')
ax.set_ylabel('%')
ax.set_ylim(0, 35)
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=4, framealpha=0.9, edgecolor='lightgray', fontsize=10)
plt.tight_layout()
plt.subplots_adjust(bottom=0.2)
plt.show()

### 유동인구 Top 20 상권

In [ ]:
# 상권 간 비교 분석
# Top N 상권 랭킹
import matplotlib.font_manager as fm

# 한글 폰트 설정 (환경에 맞게 조정)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 상권별 평균 유동인구 (분기 여러 개 있으므로 평균 사용)
top_n = (df.groupby('상권_코드_명')['총_유동인구_수']
           .mean()
           .sort_values(ascending=False)
           .head(20))

top_n.sort_values().plot(kind='barh', figsize=(10, 8))
plt.xlabel('평균 총 유동인구 수')
plt.title('유동인구 Top 20 상권')
plt.tight_layout()
plt.show()

In [ ]:
types = df['상권_구분_코드_명'].unique()

color_dict = {
    '골목상권':  '#5DCAA5',
    '발달상권':  '#378ADD',
    '관광특구':  '#D85A30',
    '전통시장':  '#EF9F27'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, t in enumerate(sorted(types)):
    top = (df[df['상권_구분_코드_명'] == t]
             .groupby('상권_코드_명')['총_유동인구_수']
             .mean()
             .sort_values(ascending=False)
             .head(10)
             .sort_values())  # barh는 아래가 1위라 오름차순

    color = color_dict.get(t, '#888')
    top.plot(kind='barh', ax=axes[i], color=color)
    axes[i].set_title(f'{t} Top 10', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('평균 총 유동인구 수')
    axes[i].set_ylabel('')

        # 막대 끝에 값 표시
    for bar in axes[i].patches:
        axes[i].text(
            bar.get_width() * 1.01,
            bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width()/1e6:.1f}M',
            va='center', fontsize=9
        )

plt.suptitle('상권 유형별 Top 10 상권 유동인구', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 상권 유형별 x 요일별 유동인구 비율

In [ ]:
day_cols_raw = [
    '월요일_유동인구_수', '화요일_유동인구_수', '수요일_유동인구_수',
    '목요일_유동인구_수', '금요일_유동인구_수', '토요일_유동인구_수', '일요일_유동인구_수'
]
day_labels = ['월', '화', '수', '목', '금', '토', '일']

# 요일별 비율 생성
for col, label in zip(day_cols_raw, day_labels):
    df[f'{label}요일_비율'] = df[col] / df['총_유동인구_수'] * 100

day_ratio_cols = [f'{l}요일_비율' for l in day_labels]

# 평일·주말 파생 컬럼
df['평일_평균_비율'] = df[[f'{l}요일_비율' for l in ['월','화','수','목','금']]].mean(axis=1)
df['주말_평균_비율'] = df[[f'{l}요일_비율' for l in ['토','일']]].mean(axis=1)
df['주말_집중도']   = df['주말_평균_비율'] - df['평일_평균_비율'] 

In [ ]:
day_by_type = df.groupby('상권_구분_코드_명')[day_ratio_cols].mean()
day_by_type.columns = day_labels

colors = {
    '골목상권': '#7F77DD', '관광특구': '#378ADD',
    '발달상권': '#1D9E75', '전통시장': '#EF9F27'
}

x = np.arange(len(day_labels))
width = 0.2
fig, ax = plt.subplots(figsize=(11, 5))

for i, (type_name, row) in enumerate(day_by_type.iterrows()):
    bars = ax.bar(x + i * width, row.values, width,
                  label=type_name, color=colors[type_name],
                  edgecolor='white', linewidth=0.5)
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8,
                 color=colors[type_name])

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(day_labels, fontsize=12)
ax.set_ylim(0, 25)
ax.set_title('상권 유형별 x 요일별 유동인구 비율')
ax.set_ylabel('%')
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=4, framealpha=0.9, edgecolor='lightgray', fontsize=10)
plt.tight_layout()
plt.subplots_adjust(bottom=0.2)
plt.show()

# 코로나 시기 비교 

### 관광특구 - 가장 크게 떨어지고, 가장 많이 회복
### 엔데믹 구간에서 회복했지만 코로나 후에 감소하는 현상 발견 → . 외국인 관광객 회복이 기대에 못 미쳤거나, 사회적 이슈나로 인해 이미지 타격

### 골목상권
### 골목상권 자체의 구조적 감소(꼭 코로나의 문제는 아닌걸로 보임) → 온라인 소비 전환, 대형몰 집중화 영향

### 발달상권
### 엔데믹 이후 회복이 거의 stop
### → 직장인 유동인구가 줄어든 영향

### 전통시장
### 전반적으로 비중이 가장 작음

In [ ]:
# 코로나 구간 매핑 함수
def covid_label(code):
    if 20191 <= code <= 20194:
        return '①코로나 전'
    elif 20201 <= code <= 20223:
        return '②코로나 집중'
    elif 20224 <= code <= 20232:
        return '③엔데믹'
    elif 20233 <= code <= 20254:
        return '④코로나 후'
    else:
        return '기타'

df['코로나_구간'] = df['기준_년분기_코드'].apply(covid_label)

# 구간 순서 고정 (그래프 정렬용)
covid_order = ['①코로나 전', '②코로나 집중', '③엔데믹', '④코로나 후']
df['코로나_구간'] = pd.Categorical(df['코로나_구간'],
                                    categories=covid_order, ordered=True)

# 구간 × 상권 유형별 평균 유동인구
covid_by_type = (df.groupby(['코로나_구간', '상권_구분_코드_명'])['총_유동인구_수']
                    .mean()
                    .unstack('상권_구분_코드_명'))

print(covid_by_type.map(lambda x: f'{x:,.0f}'))

In [ ]:
colors = {
    '골목상권': '#7F77DD', '관광특구': '#378ADD',
    '발달상권': '#1D9E75', '전통시장': '#EF9F27'
}

fig, ax = plt.subplots(figsize=(10, 6))  # subplot 제거, 단일 ax

x = np.arange(len(covid_order))
width = 0.2

for i, type_name in enumerate(covid_by_type.columns):
    offset = (i - len(covid_by_type.columns) / 2 + 0.5) * width
    bars = ax.bar(x + offset,
                  covid_by_type[type_name].values,
                  width, label=type_name,
                  color=colors[type_name],
                  edgecolor='white', linewidth=0.5)
    ax.bar_label(bars,
                 labels=[f'{v/1e6:.2f}M' for v in covid_by_type[type_name].values],
                 padding=3, fontsize=9, color=colors[type_name])

ax.set_xticks(x)
ax.set_xticklabels(covid_order, fontsize=11)
ax.set_title('코로나 구간별 평균 유동인구', fontsize=13)
ax.set_ylabel('평균 유동인구 수')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()